In [ ]:
# Hopsworks-Projektverbindungsskript
import os
from pathlib import Path

import hopsworks
from dotenv import load_dotenv

# Sicheres Laden der .env-Datei aus dem Projektstamm
project_root = Path.cwd().resolve()
if not (project_root / ".env").exists():
    project_root = project_root.parent

load_dotenv(project_root / ".env")

# Umgebungsvariablen abrufen
api_key = os.getenv("HOPSWORKS_API_KEY")
project_name = os.getenv("HOPSWORKS_PROJECT_NAME")

if not api_key or not project_name:
    raise ValueError(
        "HOPSWORKS_API_KEY und HOPSWORKS_PROJECT_NAME müssen in der .env-Datei gesetzt sein."
    )

# Verbindung zum Hopsworks-Projekt herstellen
project = hopsworks.login(
    api_key_value=api_key,
    project=project_name,
    host="eu-west.cloud.hopsworks.ai",
    port=443,
)

print(f"✅ Erfolgreich verbunden mit Projekt: {project.name}")

In [3]:
# Rohdaten via API abrufen
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry

# Session mit Cache & Retry aufsetzen
cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# API-Parameter definieren
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 47.3769,      # z.B. Zürich
    "longitude": 8.5417,
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "wind_speed_10m",
        "surface_pressure"
    ],
    "past_days": 7,
    "forecast_days": 1,
}

response = openmeteo.weather_api(url, params=params)[0]

# Stündliche Daten extrahieren
hourly = response.Hourly()
start_ts = hourly.Time()
num_points = len(hourly.Variables(0).ValuesAsNumpy())
start_dt = pd.to_datetime(start_ts, unit="s", utc=True)

dates = pd.date_range(start=start_dt, periods=num_points, freq="h")

hourly_data = {
    "date": dates,
    "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
    "relative_humidity_2m": hourly.Variables(1).ValuesAsNumpy(),
    "precipitation": hourly.Variables(2).ValuesAsNumpy(),
    "wind_speed_10m": hourly.Variables(3).ValuesAsNumpy(),
    "surface_pressure": hourly.Variables(4).ValuesAsNumpy(),
}

df_raw = pd.DataFrame(hourly_data)
print(df_raw.head())
print(df_raw.shape)

                       date  temperature_2m  relative_humidity_2m  \
0 2026-09-07 00:00:00+00:00       18.584000                  82.0   
1 2026-09-07 01:00:00+00:00       18.133999                  85.0   
2 2026-09-07 02:00:00+00:00       17.584000                  84.0   
3 2026-09-07 03:00:00+00:00       17.233999                  84.0   
4 2026-09-07 04:00:00+00:00       17.383999                  80.0   

   precipitation  wind_speed_10m  surface_pressure  
0            0.0        2.595997        975.370178  
1            0.0        3.976330        975.203247  
2            0.0        4.213692        974.925049  
3            0.0        1.800000        975.250244  
4            0.0        2.414953        976.608887  
(192, 6)


In [ ]:
# Access the feature store
fs = project.get_feature_store()